# 📥 06. ANFIS Preparation (6-Feature Input)

### 🎯 Objective: Prepare ANFIS dataset with 6-feature input (3 soft membership + 3 deltas) and targets.

### 📤 Output Artifacts: `../../data/processed/6_anfis_dataset.csv`


In [1]:
import pandas as pd

INPUT_PATH = '../../data/processed/5_clustered_telemetry.csv'
OUTPUT_PATH = '../../data/processed/6_anfis_dataset.csv'

print("Libraries imported")

Libraries imported


In [2]:
# Load data
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows")

Loaded 3240 rows


# 📓 

## 💡 Technical Specification: Temporal Deltas

> **Concept:** The AI needs to know the *derivative* of behavior (Velocity), not just Position.
>
> **Formula:** `Delta(t) = SoftMembership(t) - SoftMembership(t-1)`
>
> **Why?**
> If Combat score is 0.8, are they *starting* a fight (Delta > 0) or *ending* one (Delta < 0)? The difficulty adjustment must differ. Deltas provide this context.



# 📓 

## 📜 The 'Variance Collapse' Saga (Phase 1)

> **The Failure:** In early versions, our target variable had a standard deviation of **0.011**. The ML model saw a flat line and predicted the mean (1.0) for everyone.
>
> **The Breakthrough (Option B):**
> We realized **State** is stable, but **Change** (Deltas) is volatile. By weighting the target formula heavily on **Deltas** (Coeffs 0.55/0.40), we increased variance by **5.5x** (to 0.062), finally giving the neural network a signal to learn.



# 📓 OPTION B: Canonical Formula (FINAL v2.2)
# 🤖 Target variance maximized subject to fuzzy-membership constraints
import numpy as np

# 📓 Centered soft membership (context/bias)
combat_c = df['soft_combat'] - 0.5
collect_c = df['soft_collect'] - 0.5
explore_c = df['soft_explore'] - 0.5

# 📓 Deltas (primary variance drivers)
delta_combat = df['delta_combat'].fillna(0)
delta_collect = df['delta_collect'].fillna(0)
delta_explore = df['delta_explore'].fillna(0)

# 📓 Death rate normalization
deaths = df['death_count'].fillna(0)
death_rate = deaths / (deaths.quantile(0.95) + 1.0)
death_rate = death_rate.clip(0, 1)

# 📓 Canonical target formula
df['target_multiplier'] = (
    0.9
    + 0.22 * combat_c
    + 0.18 * collect_c
    + 0.15 * explore_c
    + 0.55 * delta_combat
    + 0.40 * delta_collect
    + 0.35 * delta_explore
    - 0.25 * death_rate
)

# 📓 Conservative clamp
df['target_multiplier'] = np.clip(df['target_multiplier'], 0.6, 1.4)

# 📓 Validation stats
t = df['target_multiplier']
print('='*60)
print('Option B Target Statistics (FINAL)')
print('='*60)
print(f'Min:  {t.min():.4f}')
print(f'Max:  {t.max():.4f}')
print(f'Mean: {t.mean():.4f}')
print(f'Std:  {t.std():.4f}')
print(f'Span: {t.max()-t.min():.4f}')
print('='*60)


In [3]:
# Select 6-feature input for ANFIS
input_features = [
    'soft_combat', 'soft_collect', 'soft_explore',
    'delta_combat', 'delta_collect', 'delta_explore'
]

print("ANFIS Input features (6):")
for i, f in enumerate(input_features, 1):
    print(f"{i}. {f}")

ANFIS Input features (6):
1. soft_combat
2. soft_collect
3. soft_explore
4. delta_combat
5. delta_collect
6. delta_explore


In [4]:
# OPTION B: Canonical Formula (FINAL v2.2)
# Target variance maximized subject to fuzzy-membership constraints
import numpy as np

# Centered soft membership (context/bias)
combat_c = df['soft_combat'] - 0.5
collect_c = df['soft_collect'] - 0.5
explore_c = df['soft_explore'] - 0.5

# Deltas (primary variance drivers)
delta_combat = df['delta_combat'].fillna(0)
delta_collect = df['delta_collect'].fillna(0)
delta_explore = df['delta_explore'].fillna(0)

# Death rate normalization
deaths = df['death_count'].fillna(0)
death_rate = deaths / (deaths.quantile(0.95) + 1.0)
death_rate = death_rate.clip(0, 1)

# Canonical target formula
df['target_multiplier'] = (
    0.9
    + 0.22 * combat_c
    + 0.18 * collect_c
    + 0.15 * explore_c
    + 0.55 * delta_combat
    + 0.40 * delta_collect
    + 0.35 * delta_explore
    - 0.25 * death_rate
)

# Conservative clamp
df['target_multiplier'] = np.clip(df['target_multiplier'], 0.6, 1.4)

# Validation stats
t = df['target_multiplier']
print('='*60)
print('Option B Target Statistics (FINAL)')
print('='*60)
print(f'Min:  {t.min():.4f}')
print(f'Max:  {t.max():.4f}')
print(f'Mean: {t.mean():.4f}')
print(f'Std:  {t.std():.4f}')
print(f'Span: {t.max()-t.min():.4f}')
print('='*60)

Option B Target Statistics (FINAL)
Min:  0.6094
Max:  1.0225
Mean: 0.7989
Std:  0.0624
Span: 0.4131


In [5]:
# Save
final_cols = ['userId', 'timestamp', 'cluster'] + input_features + ['target_multiplier']
anfis_df = df[final_cols].copy()
anfis_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

Saved to ../../data/processed/6_anfis_dataset.csv
